<a href="https://colab.research.google.com/github/aldo02032004/naufaldo.github.io/blob/main/Generic_Topic_Top_Author_Sentiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#SETUP 1 - Install & Import Dependencies

This notebook classifies social media posts into custom themes,
extracts named entities (locations, institutions, people), and
ranks the most influential authors per theme using the Gemini API.
Run this cell once per Colab session to install and load all
required libraries.

In [ ]:
!pip install -q -U google-genai pandas tqdm emoji

import os            # env vars, file/path checks (e.g. checkpoint resume logic)
import re            # regex, used for keyword matching and text cleaning
import json          # parsing LLM JSON responses
import time          # sleep/backoff between API calls
import hashlib       # generating dedup keys for near-duplicate text detection
import pandas as pd  # main dataframe library
from tqdm.auto import tqdm            # progress bars for long-running loops
from IPython.display import display   # pretty dataframe rendering in notebook output
from concurrent.futures import ThreadPoolExecutor  # optional parallel API calls
from google import genai              # Gemini API client
from google.genai import types        # Gemini API config/types (e.g. response_mime_type)
from google.colab import drive, userdata  # Google Drive mount + Colab Secrets access

try:
    import emoji
    HAS_EMOJI_LIB = True
except ImportError:
    HAS_EMOJI_LIB = False
    print("[WARN] 'emoji' library not found -> emojis will be stripped instead of converted to text.")

print("Libraries ready.")

# SETUP 2 - Configure API Key
This notebook uses Google's Gemini API for theme classification,
named entity recognition (NER), and author summarization. Before running this cell:
1. Get a free Gemini API key at https://aistudio.google.com/apikey
2. In Colab, click the key icon (🔑) in the left sidebar -> "Add new secret"
3. Name it exactly: GOOGLE_API_KEY
4. Paste your key as the value, and enable notebook access for it

In [ ]:
import os
from google.colab import userdata

# Gemini (used for theme classification & NER in the previous steps)
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

print("API key has been set.")

# SETUP 3 — LOAD DATA (WITH RESUME) & SETUP PIPELINE PARAMETERS

os, re, pandas have been imported in Step 1 — not re-imported here.from config import CONFIG is specifically placed here (not in Step 1) because this is project-specific: it only becomes relevant once config.py is created from config_template.py.

In [ ]:
CONFIG = {
    # === 1. SOURCE DATA (Google Drive) ===
    "drive_xlsx_url": "https://docs.google.com/spreadsheets/d/XXXXXXXXXXXXXXXX/edit?usp=sharing",
    "sheet_name": "Sheet1",
    "header_skiprows": 1,

    # === 2. CHECKPOINT / RESUME FILE ===
    # Give a specific name per project so it doesn't get overwritten by other projects
    "checkpoint_path": "/content/drive/MyDrive/df_clean_after_theme__my_project.pkl",

    # === 3. COLUMN NAME MAPPING ===
    # Do not change the key (left), adjust the value (right) to match your spreadsheet headers
    "columns": {
        "no": "No",
        "type": "Type",
        "headline": "Headline",
        "mentions": "Mentions",
        "date": "Date",
        "link": "Link",
        "media": "Media",
        "sentiment": "Sentiment",
        "author_id": "Author",
        "followers": "Followers",
        "retweeted": "Retweeted",
        "favourited": "Favourited",
    },

    # === 4. THEMES ===
    # Always leave one "others"/catch-all bucket.
    # description = concise keyword collection (official names, nicknames, events,
    # common typos, hashtags) — this text directly becomes the LLM classification instruction.
    "themes": {
        "example_theme_a": "replace with concise keywords for theme A",
        "example_theme_b": "replace with concise keywords for theme B",
        "others": "topics outside the themes above", # sebelumnya "lainnya"
    },

    # === 5. LLM PROVIDER & MODEL ===
    "gemini_model": "gemini-3.1-flash-lite",
    "google_api_key_env": "GOOGLE_API_KEY",  # previously a typo "GOOOGLE_API_KEY"

    # === 6. RATE LIMITING / BATCHING ===
    "sleep_between_calls": 4,
    "batch_size_classify": 40,
    "batch_size_ner": 20,
    "max_workers_ner": 4,

    # === 7. TOP-AUTHOR RANKING (weights must sum to 1.0) ===
    "weight_post_count": 0.6,
    "weight_engagement": 0.25,
    "weight_followers": 0.15,
    "top_n_authors_per_theme": 10,
    "max_posts_per_author_summary": 15,
}

from config import CONFIG

# -----------------------------------------------------------------------
# INITIAL VALIDATION
# -----------------------------------------------------------------------
assert "XXXXXXXX" not in CONFIG["drive_xlsx_url"], (
    "CONFIG['drive_xlsx_url'] is still a placeholder. Edit config.py."
)
_weights_sum = (
    CONFIG["weight_post_count"] + CONFIG["weight_engagement"] + CONFIG["weight_followers"]
)
assert abs(_weights_sum - 1.0) < 1e-6, (
    f"Ranking weights must sum to 1.0, currently {_weights_sum}. Fix it in config.py."
)

# -----------------------------------------------------------------------
# MOUNT GOOGLE DRIVE
# -----------------------------------------------------------------------
from google.colab import drive
drive.mount('/content/drive')

RESUME_FROM_THEME = os.path.exists(CONFIG["checkpoint_path"])

# -----------------------------------------------------------------------
# PATH A vs PATH B: RESUME OR START FROM SCRATCH
# -----------------------------------------------------------------------
if RESUME_FROM_THEME:
    # Path A: theme classification is already completed -> load directly from
    # checkpoint, no need to redownload or reclassify.
    df_clean = pd.read_pickle(CONFIG["checkpoint_path"])
    print(f"[INFO] Checkpoint found at {CONFIG['checkpoint_path']}")
    print(f"[INFO] df_clean loaded directly, {len(df_clean)} rows. Theme classification step CAN BE SKIPPED.")
    print(df_clean["theme"].value_counts())

else:
    # Path B: no checkpoint exists -> download xlsx from Google Drive and
    # start pipeline from scratch.
    get_ipython().system('pip install -q gdown openpyxl')
    import gdown

    def extract_drive_file_id(url: str) -> str:
        """Extract file ID from a standard Google Drive share link."""
        match = re.search(r"/d/([a-zA-Z0-9_-]+)", url)
        if match:
            return match.group(1)
        match = re.search(r"[?&]id=([a-zA-Z0-9_-]+)", url)
        if match:
            return match.group(1)
        raise ValueError("Cannot find file ID from URL. Check CONFIG['drive_xlsx_url'].")

    file_id = extract_drive_file_id(CONFIG["drive_xlsx_url"])
    local_path = "data_input.xlsx"

    print("[INFO] Checkpoint not found, starting from scratch.")
    print(f"[INFO] Detected File ID: {file_id}")
    print("[INFO] Downloading .xlsx file from Google Drive...")
    gdown.download(f"https://drive.google.com/uc?id={file_id}", local_path, quiet=False)

    xls = pd.ExcelFile(local_path)
    print(f"\n[INFO] Available sheets: {xls.sheet_names}")

    df_raw = pd.read_excel(
        local_path,
        sheet_name=CONFIG["sheet_name"],
        skiprows=CONFIG["header_skiprows"],
    )

    print(f"\n[OK] {len(df_raw)} rows, {len(df_raw.columns)} columns loaded successfully")
    print("Columns:", list(df_raw.columns))
    display(df_raw.head(5))

# -----------------------------------------------------------------------
# COLUMN NAME SHORTCUTS
# -----------------------------------------------------------------------
# Always defined regardless of the path above, because the next steps (NER,
# author ranking) need these column names whether df_raw exists (Path B) or
# doesn't exist (Path A, after resume).
COL = CONFIG["columns"]
COL_NO         = COL["no"]
COL_TYPE       = COL["type"]
COL_HEADLINE   = COL["headline"]
COL_MENTIONS   = COL["mentions"]
COL_DATE       = COL["date"]
COL_LINK       = COL["link"]
COL_MEDIA      = COL["media"]
COL_SENTIMENT  = COL["sentiment"]
COL_AUTHOR_ID  = COL["author_id"]
COL_FOLLOWERS  = COL["followers"]
COL_RETWEETED  = COL["retweeted"]
COL_FAVOURITED = COL["favourited"]

# -----------------------------------------------------------------------
# THEME LIST
# -----------------------------------------------------------------------
# Taken from config.py — edit config.py to change themes, not these lines.
THEMES = list(CONFIG["themes"].keys())
THEME_DESCRIPTIONS = CONFIG["themes"]

# -----------------------------------------------------------------------
# LLM CLIENT & PARAMETERS
# -----------------------------------------------------------------------
# All values are sourced from CONFIG, nothing is hardcoded here.
from google import genai
from google.genai import types

GEMINI_MODEL = CONFIG["gemini_model"]
gemini_client = genai.Client(api_key=os.environ[CONFIG["google_api_key_env"]])

SLEEP_BETWEEN_CALLS = CONFIG["sleep_between_calls"]
BATCH_SIZE_CLASSIFY = CONFIG["batch_size_classify"]
BATCH_SIZE_NER = CONFIG["batch_size_ner"]
MAX_WORKERS_NER = CONFIG["max_workers_ner"]
TOP_N_AUTHORS_PER_THEME = CONFIG["top_n_authors_per_theme"]
MAX_POSTS_PER_AUTHOR_SUMMARY = CONFIG["max_posts_per_author_summary"]
WEIGHT_POST_COUNT = CONFIG["weight_post_count"]
WEIGHT_ENGAGEMENT = CONFIG["weight_engagement"]
WEIGHT_FOLLOWERS = CONFIG["weight_followers"]

print(f"\n[INFO] Active mode: {'RESUME (continuing from saved df_clean)' if RESUME_FROM_THEME else 'FULL (starting from scratch)'}")
print("[INFO] Configuration ready.")

### SETUP 3a — FILTER OUT MEDIA/NEWS ACCOUNTS

Runs AFTER Step 3 (load/resume) and BEFORE Step 4 (classification) to avoid wasting LLM API quota on media rows.

GENERIC FILE — `MEDIA_ACCOUNTS` contains a solid baseline of Indonesian/Regional news outlets.

⚠️ ADDING NEW MEDIA: If your project needs specific niche/foreign media filtered out, add them to `CONFIG["extra_media_accounts"]` in `config.py` instead of editing this file.

⚠️ NOTE: This step is skipped automatically if you are resuming from a checkpoint (`RESUME_FROM_THEME == True`), as media was already filtered out before saving.

In [ ]:
import re

if RESUME_FROM_THEME:
    print("[INFO] Resuming from checkpoint — media filtering already applied in a previous run, skipping Step 3a.")

else:
    MEDIA_TYPE_ALWAYS = {"news"}
    DOMAIN_SUFFIXES = (".com", ".id", ".co", ".net", ".org")
    DOMAIN_TEXT_SUFFIXES = ("dotcom", "dotid", "dotco", "dotnet", "dotorg")  # many outlets spell their domain out as text

    # -----------------------------------------------------------------------
    # MEDIA ACCOUNT WHITELIST
    # -----------------------------------------------------------------------
    # SINGLE SOURCE OF TRUTH for MEDIA_ACCOUNTS — do not redefine this set
    # elsewhere in the pipeline, or accounts can silently "disappear" if a
    # second definition overwrites this one.
    # Account names are stored normalized: lowercase, no "@", no symbols/digits
    # stripped out by normalize_account() before matching.
    MEDIA_ACCOUNTS = {
        # Indonesian national outlets
        "detikcom", "kompascom", "cnnindonesia", "tempodotco", "antaranews",
        "cnbcindonesia", "liputan6dotcom", "liputan6", "sctv", "tvonenews", "kumparan",
        "beritasatu", "metrotvnews", "tribunnews", "republikaonline", "suaradotcom",
        "jawapos", "bbcindonesia", "voaindonesia", "narasitv", "sindonews",
        "vivacoid", "bisniscom", "hariankompas", "tirtoid", "alineadotid",
        "inilahcom", "merdekadotcom", "okezone", "idntimes", "medcomid",
        "rri", "tvri", "jpnndotcom", "grid", "inewsdotid", "fajar",
        "pikiranrakyat", "gatra", "nuonline", "radioelshinta", "mediaindonesia",
        "setkabgoid",
        # official Cabinet Secretariat account -> government media, not a personal account
        # Malaysian outlets
        "awani", "bharianmy", "bernamadotcom", "utusandotcom", "sinarharian",
        "malaysiakini", "thestar", "nst", "theedgemarkets",
        # Russian state/international outlets — included by default since Russia
        # is a common subject in Indonesian foreign-affairs coverage; keep or
        # trim this block depending on your topic.
        "rt", "actualidadrt", "rtcom", "sputnik", "sputniknews", "sputnikindonesia",
        "tass", "tassagency", "rianovosti", "ria", "kremlin", "kremlinru",
        "telesur", "telesurtv",
    }

    # Merge in any project-specific extra media accounts defined in config.py.
    # Define CONFIG["extra_media_accounts"] as a set/list of normalized account
    # names (lowercase, no "@", no digits/symbols) if your topic needs outlets
    # not covered above. Defaults to empty if not set.
    MEDIA_ACCOUNTS |= set(CONFIG.get("extra_media_accounts", []))

    MEDIA_KEYWORDS_SUBSTRING = [
        "news", "media", "redaksi", "newsroom", "official", "humas", "koran",
        "awani", "gazette", "tv", "radio", "pers",
        # NOTE: government institution keywords (kemen, bnpb, bmkg, bpbd, polri,
        # setneg) are DELIBERATELY excluded here -> ministry/agency accounts
        # still count as candidate "top authors", not flagged as "media".
        # NOTE: "pers" as a substring can false-positive on names like "persija",
        # "persib" (football clubs) -> if your data covers football, move "pers"
        # into MEDIA_KEYWORDS_EXACT below instead.
    ]
    # Short/ambiguous keywords that need exact-token matching instead of
    # substring matching -> currently empty, add here if needed for your data.
    MEDIA_KEYWORDS_EXACT = []

    def normalize_account(text: str) -> str:
        """Lowercase + alphanumeric only, for matching against the whitelist/domain
        suffixes. IMPORTANT: keep digits -> many outlet names include numbers,
        e.g. 'liputan6dotcom'."""
        return re.sub(r"[^a-z0-9]", "", str(text).strip().lower())

    def contains_media_keyword(author_lower: str) -> bool:
        # Check substring keywords first (safe/specific ones, including tv/radio/pers).
        if any(kw in author_lower for kw in MEDIA_KEYWORDS_SUBSTRING):
            return True
        # Exact-token check for any remaining ambiguous keywords (currently empty).
        tokens = re.split(r"[^a-z]+", author_lower)
        return any(tok in MEDIA_KEYWORDS_EXACT for tok in tokens if tok)

    def is_media_account(row) -> bool:
        """Return True if this row's author should be treated as a media/news
        account rather than an individual/organic poster."""
        media_val = str(row.get(COL_MEDIA, "")).strip().lower()
        author_raw = str(row.get(COL_AUTHOR_ID, "")).strip().lower().lstrip("@")
        author_norm = normalize_account(author_raw)

        if media_val in MEDIA_TYPE_ALWAYS:
            return True
        if author_norm in MEDIA_ACCOUNTS:
            return True
        if contains_media_keyword(author_raw):
            return True
        if author_raw.endswith(DOMAIN_SUFFIXES):
            return True
        if author_norm.endswith(DOMAIN_TEXT_SUFFIXES):
            return True
        return False

    # -----------------------------------------------------------------------
    # APPLY FILTER
    # -----------------------------------------------------------------------

    # Make sure Followers is numeric first, so sort_values() in the QC checks
    # below is accurate.
    df_raw[COL_FOLLOWERS] = pd.to_numeric(df_raw.get(COL_FOLLOWERS, 0), errors="coerce").fillna(0)

    # Keep a full copy of the raw data, in case you need to re-check filtered
    # rows later.
    df_raw_original = df_raw.copy()

    df_raw["is_media"] = df_raw.apply(is_media_account, axis=1)

    n_media = df_raw["is_media"].sum()
    n_nonmedia = (~df_raw["is_media"]).sum()
    print(f"[INFO] {n_media} rows flagged as media (dropped BEFORE cleaning)")
    print(f"[INFO] {n_nonmedia} non-media rows remaining (proceeding to cleaning)")

    print("\nSample detection per unique account:")
    display(
        df_raw[[COL_AUTHOR_ID, COL_MEDIA, "is_media"]]
        .drop_duplicates(subset=COL_AUTHOR_ID)
        .head(20)
    )

    print("\n[QC 1] Highest-follower accounts FLAGGED as media — check none of these are actually a large personal/influencer account wrongly caught by the filter:")
    display(
        df_raw[df_raw["is_media"]]
        .drop_duplicates(subset=COL_AUTHOR_ID)
        .sort_values(COL_FOLLOWERS, ascending=False)
        [[COL_AUTHOR_ID, COL_MEDIA, COL_FOLLOWERS]]
        .head(15)
    )

    print("\n[QC 2] Highest-follower accounts NOT flagged — check none of these are actually a media outlet that slipped past the heuristic:")
    display(
        df_raw[~df_raw["is_media"]]
        .drop_duplicates(subset=COL_AUTHOR_ID)
        .sort_values(COL_FOLLOWERS, ascending=False)
        [[COL_AUTHOR_ID, COL_MEDIA, COL_FOLLOWERS]]
        .head(15)
    )

    df_raw = df_raw[~df_raw["is_media"]].drop(columns=["is_media"]).reset_index(drop=True)
    print(f"\n[INFO] df_raw now contains {len(df_raw)} non-media rows, ready for cleaning")

### SETUP 3b — TEXT CLEANING

Runs after media filtering (3a). Combines 'Headline' + 'Mentions', then cleans the text (removes URLs/mentions, fixes letter elongation, converts emojis, and normalizes Indonesian slang). It also generates a key for deduplicating retweets.

⚠️ LANGUAGE WARNING: The `KAMUS_ALAY` dictionary is strictly for Bahasa Indonesia. Replace it if processing other languages.

⚠️ CUSTOM SLANG: To add project-specific abbreviations or slang, use `CONFIG["extra_kamus_alay"]` in `config.py` instead of editing this base dictionary.

In [ ]:
import re
import hashlib

# -----------------------------------------------------------------------
# INDONESIAN SLANG ("ALAY") NORMALIZATION DICTIONARY
# -----------------------------------------------------------------------
KAMUS_ALAY = {
    "gak": "tidak", "ga": "tidak", "nggak": "tidak", "tdk": "tidak", "gk": "tidak",
    "kaga": "tidak", "kagak": "tidak", "ngga": "tidak", "enggak": "tidak", "tak": "tidak",
    "gaada": "tidak ada", "bgt": "banget", "bnget": "banget", "bgttt": "banget",
    "bgt2": "banget", "banget2": "banget", "sangattt": "sangat",
    "yg": "yang", "yng": "yang", "krn": "karena", "krna": "karena", "karna": "karena",
    "sm": "sama", "sm2": "sama-sama", "sama2": "sama-sama",
    "utk": "untuk", "u/": "untuk", "bwt": "buat", "buatt": "buat", "utuk": "untuk",
    "dr": "dari", "drpd": "daripada",
    "skrg": "sekarang", "skrng": "sekarang", "skarang": "sekarang", "skg": "sekarang",
    "dgn": "dengan", "dg": "dengan", "dngan": "dengan",
    "org": "orang", "orng": "orang",
    "tp": "tapi", "tpi": "tapi", "spt": "seperti", "kayak": "seperti", "kaya": "seperti", "kyk": "seperti",
    "blm": "belum", "jgn": "jangan", "jgnkan": "jangankan", "jgnlah": "janganlah",
    "emg": "memang", "emang": "memang", "emank": "memang", "emgnya": "memangnya", "emangnya": "memangnya",
    "jd": "jadi", "jadinya": "jadinya", "jg": "juga", "aja": "saja", "aj": "saja",
    "udah": "sudah", "udh": "sudah", "dah": "sudah", "sdh": "sudah",
    "gmn": "bagaimana", "gmna": "bagaimana", "gmana": "bagaimana", "gimana": "bagaimana",
    "knp": "kenapa", "knpa": "kenapa", "napa": "kenapa", "ngapain": "sedang apa", "ngapa": "kenapa",
    "kalo": "kalau", "klo": "kalau",
    "trs": "terus", "trus": "terus", "gini": "begini", "gitu": "begitu", "gt": "begitu", "gtu": "begitu",
    "sy": "saya", "km": "kamu", "gw": "saya", "gwa": "saya", "gua": "saya", "gue": "saya",
    "ane": "saya", "aq": "saya", "aku": "saya",
    "lo": "kamu", "lu": "kamu", "elu": "kamu", "elo": "kamu", "ente": "kamu", "situ": "kamu",
    "hrs": "harus", "harus2": "harus", "kudu": "harus", "musti": "harus",
    "bs": "bisa", "bisa2": "bisa", "msh": "masih", "dlm": "dalam",
    "sblm": "sebelum", "stlh": "setelah", "pd": "pada", "pgn": "ingin", "pengen": "ingin",
    "liat": "lihat", "abis": "habis", "bkn": "bukan",
    "gpp": "tidak apa-apa", "gapapa": "tidak apa-apa",
    "cmn": "cuma", "cuman": "cuma", "mksh": "terima kasih", "makasih": "terima kasih",
    "moga": "semoga", "smoga": "semoga",
    "wkt": "waktu", "cpt": "cepat", "lg": "lagi", "lgi": "lagi",
    "sll": "selalu", "sllu": "selalu", "prnh": "pernah",
    "sndiri": "sendiri", "stiap": "setiap", "byk": "banyak", "dikit": "sedikit",
    "dtg": "datang", "krg": "kurang", "ato": "atau",
    "pke": "pakai", "pake": "pakai", "denger": "dengar", "kasih": "beri", "bikin": "buat",
    "brp": "berapa", "walopun": "walaupun", "walaupun": "walaupun", "meskipun": "meskipun", "meski": "meskipun",
    "kayanya": "sepertinya", "kykny": "sepertinya",
    "dpt": "dapat", "dapet": "dapat", "tggl": "tinggal", "tinggl": "tinggal",
    "tmn": "teman", "temen": "teman",
    "trnyata": "ternyata", "ternyta": "ternyata",
    "sbnrnya": "sebenarnya", "sebenernya": "sebenarnya", "sbnernya": "sebenarnya",
    "sbg": "sebagai", "sbgai": "sebagai", "trhdp": "terhadap", "thd": "terhadap", "thdp": "terhadap",
    "diantaranya": "di antaranya", "diantara": "di antara",
    "ngerti": "mengerti", "ngerasa": "merasa", "berasa": "terasa",
    "keliatan": "terlihat", "keliatannya": "terlihatnya",
    "nyari": "mencari", "nyoba": "mencoba", "nunggu": "menunggu",
    "ngasih": "memberi", "ngajak": "mengajak", "ngobrol": "berbicara",
    "nyadar": "sadar", "ngerasain": "merasakan", "ngebayangin": "membayangkan",
    "mikir": "berpikir", "mikirin": "memikirkan", "ngomongin": "membicarakan",
    "kesel": "kesal", "sebel": "sebal", "capek": "lelah", "cape": "lelah",
    "males": "malas", "mager": "malas gerak", "seneng": "senang",
    "parah": "sangat", "gila": "sangat", "anjir": "sangat", "anjay": "sangat",
    "mantap": "bagus", "mantul": "bagus", "keren": "bagus",
    "jelek": "buruk", "ancur": "hancur", "hancur": "hancur",
    "ngeri": "mengerikan", "serem": "menyeramkan",
    "bego": "bodoh", "goblok": "bodoh", "tolol": "bodoh", "bodo": "bodoh",
    "songong": "sombong", "belagu": "sombong",
    "curhat": "curahan hati", "japri": "pesan pribadi",
    "bener": "benar", "beneran": "benaran", "makanya": "makanya", "makannya": "makanya",
    "makin": "semakin", "kian": "semakin",
    "besok": "besok", "bsk": "besok", "kmrn": "kemarin", "kmarin": "kemarin", "kemaren": "kemarin",
    "td": "tadi", "entar": "nanti", "ntar": "nanti", "nti": "nanti",
    "pemrintah": "pemerintah", "pemerintahan": "pemerintah",
    "korup": "korupsi", "dikorupsi": "korupsi", "ngorupsi": "korupsi",
    "nyolong": "mencuri", "maling": "pencuri",
    "boong": "bohong", "bohong2": "bohong", "hoax": "hoaks", "hoak": "hoaks",
    "settingan": "rekayasa", "settingannya": "rekayasa", "php": "janji palsu",
    "ngamuk": "marah", "murka": "marah", "demo": "demonstrasi",
    "smpai": "sampai", "sampe": "sampai",
    "tuhh": "tuh", "sihh": "sih", "dehh": "deh",
    "pak": "bapak", "bu": "ibu", "min": "admin",
}

# Merge in any project-specific slang/abbreviations defined in config.py.
# Define CONFIG["extra_kamus_alay"] as a dict of {slang: standard_form} if
# your topic has jargon/abbreviations not covered above (e.g. domain-specific
# acronyms your audience uses casually). Defaults to empty if not set.
KAMUS_ALAY.update(CONFIG.get("extra_kamus_alay", {}))

# -----------------------------------------------------------------------
# REGEX PATTERNS
# -----------------------------------------------------------------------
RE_URL = re.compile(r"(https?://\S+|www\.\S+)")
RE_MENTION = re.compile(r"@[A-Za-z0-9_]+")
RE_HASHTAG = re.compile(r"#([A-Za-z0-9_]+)")
RE_RT_PREFIX = re.compile(r"^\s*RT\s*@[A-Za-z0-9_]+\s*:\s*", flags=re.IGNORECASE)
RE_ELONGATION = re.compile(r"(.)\1{2,}")
RE_MULTI_SPACE = re.compile(r"\s+")
RE_NON_ALNUM_PUNCT = re.compile(r"[^\w\s.,!?]")


def demojize_or_strip(text: str) -> str:
    """Convert emoji to words (e.g. 😀 -> "grinning face") if the `emoji`
    library is available; otherwise emoji characters get stripped out later
    by RE_NON_ALNUM_PUNCT instead. HAS_EMOJI_LIB is set in Step 1."""
    if HAS_EMOJI_LIB:
        text = emoji.demojize(text, language="id" if "id" in emoji.LANGUAGES else "en")
        return text.replace("_", " ").replace(":", " ")
    return text


def fix_elongation(text: str) -> str:
    """Collapse repeated characters down to 2, e.g. "parahhhh" -> "parahh".
    (Kept at 2 rather than 1 to preserve legitimate double letters.)"""
    return RE_ELONGATION.sub(r"\1\1", text)


def normalize_slang(text: str) -> str:
    """Replace Indonesian slang/abbreviations word-by-word using KAMUS_ALAY.
    Words not found in the dictionary are left unchanged."""
    words = text.split()
    return " ".join(KAMUS_ALAY.get(w.lower().strip(".,!?"), w) for w in words)


def extract_hashtags(text: str):
    """Return a list of hashtags found in the text (without the '#')."""
    return RE_HASHTAG.findall(text)


def extract_mentions(text: str):
    """Return a list of @mentions found in the text (with the '@')."""
    return RE_MENTION.findall(text)


def build_raw_text(headline, mentions) -> str:
    """Combine Headline + Mentions into a single raw text.
    Avoids duplicating content when both columns hold the same text."""
    headline = "" if pd.isna(headline) else str(headline).strip()
    mentions = "" if pd.isna(mentions) else str(mentions).strip()
    if not headline or headline.lower() == mentions.lower() or headline.lower() == "nan":
        return mentions or headline
    if not mentions:
        return headline
    return f"{headline}. {mentions}"


def clean_text(raw: str) -> str:
    """Full cleaning pipeline for one text: strip RT prefix, URLs, mentions;
    unwrap hashtags to plain words; convert emoji to text; strip remaining
    punctuation/symbols; fix letter elongation; normalize slang; collapse
    whitespace."""
    if not isinstance(raw, str) or not raw.strip():
        return ""
    text = raw
    text = RE_RT_PREFIX.sub("", text)
    text = RE_URL.sub(" ", text)
    text = RE_MENTION.sub(" ", text)
    text = RE_HASHTAG.sub(r"\1", text)
    text = demojize_or_strip(text)
    text = RE_NON_ALNUM_PUNCT.sub(" ", text)
    text = fix_elongation(text)
    text = normalize_slang(text)
    text = RE_MULTI_SPACE.sub(" ", text).strip()
    return text


def make_dedup_key(text_clean: str) -> str:
    """Build a hash key from cleaned text (lowercased, punctuation stripped)
    for detecting identical or near-identical posts (e.g. mass retweets)."""
    key = re.sub(r"[^\w\s]", "", text_clean.lower())
    key = RE_MULTI_SPACE.sub(" ", key).strip()
    return hashlib.md5(key.encode("utf-8")).hexdigest()


print("Text cleaning functions ready.")

### SETUP 3c — APPLY TEXT CLEANING

Applies the cleaning functions from Step 3b to the dataframe to produce `df_clean`.

GENERIC FILE — Relies entirely on `CONFIG` column mappings. No edits needed for new projects.

⚠️ NOTE: This step is skipped automatically if resuming from a checkpoint (`RESUME_FROM_THEME == True`), as the text was already cleaned in the previous run.

In [ ]:
if RESUME_FROM_THEME:
    print("[INFO] Resuming from checkpoint — text cleaning already applied in a previous run, skipping Step 3c.")

else:
    # -----------------------------------------------------------------------
    # BUILD COMBINED RAW TEXT (Headline + Mentions)
    # -----------------------------------------------------------------------
    # Falls back to an empty-string column if either source column is
    # missing from this dataset entirely, rather than raising a KeyError.
    headline_col = df_raw[COL_HEADLINE] if COL_HEADLINE in df_raw.columns else pd.Series([""] * len(df_raw))
    mentions_col = df_raw[COL_MENTIONS] if COL_MENTIONS in df_raw.columns else pd.Series([""] * len(df_raw))
    df_raw["text_raw_combined"] = [build_raw_text(h, m) for h, m in zip(headline_col, mentions_col)]

    # -----------------------------------------------------------------------
    # EXTRACT HASHTAGS / MENTIONS, THEN CLEAN TEXT
    # -----------------------------------------------------------------------
    tqdm.pandas(desc="Cleaning text")
    df_raw["hashtags"] = df_raw["text_raw_combined"].astype(str).apply(extract_hashtags)
    df_raw["mentions_akun"] = df_raw["text_raw_combined"].astype(str).apply(extract_mentions)
    df_raw["text_clean"] = df_raw["text_raw_combined"].astype(str).progress_apply(clean_text)

    # -----------------------------------------------------------------------
    # DROP EMPTY / TOO-SHORT TEXTS
    # -----------------------------------------------------------------------
    # Anything under 3 words after cleaning is unlikely to carry enough
    # signal for theme classification or NER, and is more likely noise.
    before = len(df_raw)
    df_clean = df_raw[df_raw["text_clean"].str.split().str.len().fillna(0) >= 3].copy()
    print(f"[INFO] Dropped {before - len(df_clean)} empty/too-short rows after cleaning")

    # -----------------------------------------------------------------------
    # DEDUPLICATE IDENTICAL / RETWEETED POSTS
    # -----------------------------------------------------------------------
    df_clean["dedup_key"] = df_clean["text_clean"].apply(make_dedup_key)
    before = len(df_clean)
    df_clean["is_duplicate"] = df_clean.duplicated(subset="dedup_key", keep="first")
    n_dup = df_clean["is_duplicate"].sum()
    df_clean = df_clean[~df_clean["is_duplicate"]].drop(columns=["is_duplicate", "dedup_key"])
    df_clean = df_clean.reset_index(drop=True)  # clean 0..n index, relied on by later steps

    print(f"[INFO] Dropped {n_dup} identical/retweet duplicates")
    print(f"[OK] {len(df_clean)} rows remaining after cleaning\n")

    print("Before vs after cleaning example:")
    display(df_clean[["text_raw_combined", "text_clean"]].head(5))

# SETUP 4 — THEME CLASSIFICATION (keyword prefilter + LLM few-shot, multi-label)

GENERIC FILE — but requires manual setup for few-shot prompting.

⚠️ ACTION REQUIRED: The `FEWSHOT_THEME_EXAMPLES` below are placeholders. You MUST replace them with real text examples matching your specific topic and `CONFIG["themes"]`. Leaving irrelevant examples will confuse the LLM and degrade accuracy.

Aim for 1-2 examples per theme, ensuring you include negative controls ("others") and casual/slang variations.

The rest of the prompt-building and classification logic reads directly from `CONFIG` and requires no edits.

In [ ]:
import re
import json
import time

from config import CONFIG
# Also assumes THEMES, THEME_DESCRIPTIONS, gemini_client, GEMINI_MODEL
# are already defined by step3_load_and_configure.py


# -----------------------------------------------------------------------
# KEYWORD PREFILTER
# -----------------------------------------------------------------------
def keyword_match_theme(text):
    """Fast keyword-based prefilter using CONFIG["themes"] descriptions,
    run BEFORE calling the LLM (saves API calls for obvious cases).

    Returns a SINGLE theme name if a keyword matches, or None if nothing
    matches — in which case the text is sent to the LLM classifier instead.

    NOTE: this only ever returns one theme, even if a text could arguably
    belong to multiple themes — it's meant as a cheap filter, not the final
    multi-label answer. The LLM step handles multi-label classification.
    """
    text_lower = str(text).lower()
    for theme in THEMES:
        if theme == "others":  # skip the catch-all/"other" bucket
            continue
        desc = THEME_DESCRIPTIONS[theme]
        kw_list = [re.escape(k.strip()) for k in desc.split(",") if k.strip()]
        pattern = "|".join(kw_list)
        if re.search(pattern, text_lower):
            return theme
    return None  # no keyword matched -> let the LLM decide


THEME_LIST_STR = "\n".join(f"- {t}: {THEME_DESCRIPTIONS[t]}" for t in THEMES)


# -----------------------------------------------------------------------
# FEW-SHOT EXAMPLES — ⚠️ EDIT THIS FOR YOUR OWN PROJECT ⚠️
# -----------------------------------------------------------------------
# Template structure only. Replace every example below with real texts and
# theme labels from YOUR project (matching the theme names you defined in
# CONFIG["themes"] in config.py). Keep a good mix of:
#   - formal/news-style texts
#   - casual/social-media-style texts (slang, typos, sarcasm)
#   - multi-theme examples (a text that legitimately belongs to 2+ themes)
#   - "others"/other examples, including ones that superficially LOOK
#     related but aren't (helps the model avoid false positives)
FEWSHOT_THEME_EXAMPLES = """
Example classifications (learn the pattern from these):

Text: "[REPLACE: a formal/news-style text clearly matching theme_a]"
Answer: {"themes": ["example_theme_a"], "primary_theme": "example_theme_a", "confidence": 0.9}

Text: "[REPLACE: a text matching theme_b instead]"
Answer: {"themes": ["example_theme_b"], "primary_theme": "example_theme_b", "confidence": 0.9}

"""
# TIP: keep adding real examples here as you review misclassifications —
# this is exactly the kind of thing that could later be moved into a
# growing few-shot bank (like the ner_library.py pattern discussed earlier)
# instead of a fixed static block.


# -----------------------------------------------------------------------
# PROMPT BUILDER
# -----------------------------------------------------------------------
def build_theme_prompt(batch_texts):
    """Build the full classification prompt for a batch of texts.
    Fully generic: pulls theme list from THEME_LIST_STR (derived from
    CONFIG["themes"]) and few-shot examples from FEWSHOT_THEME_EXAMPLES.
    """
    numbered = "\n".join(f"{i+1}. {t}" for i, t in enumerate(batch_texts))
    return f"""You are a theme classifier for Indonesian tweets/news.

Available themes list:
{THEME_LIST_STR}

{FEWSHOT_THEME_EXAMPLES}

Now classify EACH text below. ONE text MAY have more than 1 theme if relevant (see examples above), but still determine the "primary_theme" as the most dominant/main theme discussed. If it doesn't fit any theme other than "others", just use "others".

Text:
{numbered}

Answer ONLY with a JSON array (no explanations, no markdown code blocks), the format is:
[
  {{"index": 1, "themes": ["example_theme_a", "example_theme_b"], "primary_theme": "example_theme_a", "confidence": 0.9}},
  {{"index": 2, "themes": ["others"], "primary_theme": "others", "confidence": 0.6}}
]
The number of items MUST exactly match the number of texts above ({len(batch_texts)} items)."""


# -----------------------------------------------------------------------
# BATCH CLASSIFICATION CALL (with retry/backoff)
# -----------------------------------------------------------------------
def classify_theme_batch(batch_texts, retry=6):
    """Send one batch to the LLM and parse the JSON response.
    Retries with backoff on quota-exhaustion / overload / transient errors.
    On total failure, falls back to labeling everything "others" with
    confidence 0.0, so the pipeline doesn't crash on one bad batch.
    """
    prompt = build_theme_prompt(batch_texts)
    for attempt in range(retry):
        try:
            resp = gemini_client.models.generate_content(
                model=GEMINI_MODEL,
                contents=prompt,
                config=types.GenerateContentConfig(
                    response_mime_type="application/json",
                    max_output_tokens=2000,
                ),
            )
            parsed = json.loads(resp.text.strip())
            if len(parsed) != len(batch_texts):
                raise ValueError(f"Number of results ({len(parsed)}) != number of inputs ({len(batch_texts)})")
            return parsed
        except Exception as e:
            err_str = str(e)
            is_quota_exhausted = "RESOURCE_EXHAUSTED" in err_str or "429" in err_str
            is_overload = "503" in err_str or "UNAVAILABLE" in err_str

            if is_quota_exhausted:
                # Daily quota errors (GenerateRequestsPerDayPerProjectPerModel)
                # need a much longer wait than a simple short backoff.
                wait = 60 * (attempt + 1)
            elif is_overload:
                wait = 20 * (attempt + 1)
            else:
                wait = 5 * (attempt + 1)

            print(f"[WARN] Theme classification batch failed (attempt {attempt+1}/{retry}): {e} -> waiting {wait}s")
            time.sleep(wait)

    print(f"[ERROR] This batch failed completely after {retry} attempts, labeled 'others' as fallback")
    return [
        {"index": i + 1, "themes": ["others"], "primary_theme": "others", "confidence": 0.0}
        for i in range(len(batch_texts))
    ]


print("Theme classification functions ready (keyword prefilter + LLM few-shot multi-label).")

### STEP 4a — ADVANCED KEYWORD PREFILTER (OPTIONAL)
Alternative to Step 4's simple `keyword_match_theme()`. Use only one; if enabling this, remove Step 4's version.

⚠️ PATTERN WARNING: This prefilter enforces an actor + context co-occurrence logic (e.g., specific person + place/event). It is ideal for "public figure involvement" topics, but may not fit other themes. Adapt it to your project or skip it to rely on Step 4's simpler substring filter.

⚠️ ACTION REQUIRED: All keyword lists below are placeholders. Replace them with project-specific keywords designed for high-recall pre-filtering of obvious cases before LLM processing.

In [ ]:
# Keywords indicating the "actor" entity (e.g. a specific person/organization
# whose involvement defines theme A). Include name variants, nicknames,
# titles, hashtags, common misspellings.
KW_ACTOR = [
    "[REPLACE: actor name]", "[REPLACE: actor nickname]", "[REPLACE: actor title]",
    "[REPLACE: #actorhashtag]",
]

# Keywords indicating the "context" entity (e.g. the place/topic/event that,
# combined with the actor, defines theme A — or on its own defines theme B).
KW_CONTEXT = [
    "[REPLACE: context place/topic name]", "[REPLACE: context event name]",
    "[REPLACE: #contexthashtag]",
]

# Broader context keywords for theme B alone (context-related topics beyond
# just the core entity list above — e.g. related countries, policies, allies).
KW_CONTEXT_BROAD = KW_CONTEXT + [
    "[REPLACE: broader related keyword 1]", "[REPLACE: broader related keyword 2]",
]

# Keywords that indicate the actor is mentioned in a totally unrelated
# (e.g. domestic/internal) context -> short-circuit to "others" without
# calling the LLM. Build this list from manual sampling of your own data
# (look at what the actor gets mentioned about OTHER than your theme).
KW_UNRELATED_EXCLUDE = [
    "[REPLACE: unrelated topic keyword 1]", "[REPLACE: unrelated topic keyword 2]",
]


def _contains_any(text_lower, kw_list):
    return any(kw in text_lower for kw in kw_list)


def keyword_match_theme_advanced(text):
    """Fast co-occurrence-based prefilter, NOT sent to the LLM.
    Returns a SINGLE theme, or None if still ambiguous -> falls through to the LLM.

    NOTE: theme names below ("theme_actor_context", "theme_context_only") are
    placeholders — replace with your actual theme names from CONFIG["themes"].
    """
    text_lower = str(text).lower()

    has_actor = _contains_any(text_lower, KW_ACTOR)
    has_context = _contains_any(text_lower, KW_CONTEXT)

    # theme_actor_context: needs actor + context to co-occur in the SAME text
    if has_actor and has_context:
        return "theme_actor_context"  # ⚠️ REPLACE with your real theme name

    # theme_context_only: context appears WITHOUT the actor
    if not has_actor and _contains_any(text_lower, KW_CONTEXT_BROAD):
        return "theme_context_only"  # ⚠️ REPLACE with your real theme name

    # short-circuit: actor present, no context at all, and clearly an
    # unrelated/domestic topic -> safe to label "others" without the LLM
    if has_actor and not has_context and _contains_any(text_lower, KW_UNRELATED_EXCLUDE):
        return "others"

    return None  # still ambiguous -> let the LLM decide


print("Advanced keyword prefilter ready (co-occurrence-based version).")

# STEP 5 — RUN THEME CLASSIFICATION (hybrid: keyword prefilter, then LLM)

Processes data in batches (`CONFIG["batch_size_classify"]`) to optimize API calls. It prioritizes the advanced keyword prefilter, falling back to the LLM classifier for ambiguous items.

GENERIC FILE — Controlled by `TEST_MODE`, `N_TEST_ROWS`, batch configurations, and `CONFIG["checkpoint_path"]`. Requires no code modifications for new projects as long as preceding pipeline steps are correctly configured.

In [ ]:

TEST_MODE = False
N_TEST_ROWS = 100

# -----------------------------------------------------------------------
# SNAPSHOT THE CLEANED DATA (only created once per session)
# -----------------------------------------------------------------------
# df_clean at this point is guaranteed non-media, since media accounts were
# already dropped in Step 3a.
if "df_master_clean" not in globals():
    df_master_clean = df_clean.copy()
    print(f"[INFO] df_master_clean created, {len(df_master_clean)} rows")
else:
    print(f"[INFO] df_master_clean already exists, {len(df_master_clean)} rows (not recreated)")

if TEST_MODE:
    df_run = df_master_clean.sample(n=min(N_TEST_ROWS, len(df_master_clean)), random_state=42).reset_index(drop=True)
    print(f"[TEST MODE] Using {len(df_run)} rows out of {len(df_master_clean)} total")
else:
    df_run = df_master_clean.copy()
    print(f"[FULL MODE] Using ALL {len(df_run)} rows")

texts_run = df_run["text_clean"].fillna("").tolist()
themes_run = [None] * len(texts_run)          # primary_theme (single, used by author ranking later)
all_themes_run = [None] * len(texts_run)       # all relevant themes (list, extra insight)
confidences_run = [0.0] * len(texts_run)

# -----------------------------------------------------------------------
# PASS 1 — KEYWORD PREFILTER
# -----------------------------------------------------------------------
for i, t in enumerate(texts_run):
    match = keyword_match_theme_advanced(t)
    if match:
        themes_run[i] = match
        all_themes_run[i] = [match]
        confidences_run[i] = 1.0  # exact keyword match, treated as high confidence

n_keyword = sum(1 for th in themes_run if th is not None)
need_llm_idx = [i for i, th in enumerate(themes_run) if th is None]
print(f"[INFO] {n_keyword} rows resolved directly by keyword prefilter")
print(f"[INFO] {len(need_llm_idx)} rows sent to the LLM (no keyword match)")

# -----------------------------------------------------------------------
# PASS 2 — LLM CLASSIFICATION (few-shot, multi-label) FOR THE REST
# -----------------------------------------------------------------------
texts_for_llm = [texts_run[i] for i in need_llm_idx]

n_batches = (len(texts_for_llm) + BATCH_SIZE_CLASSIFY - 1) // BATCH_SIZE_CLASSIFY
for b in tqdm(range(n_batches), desc="Theme classification (LLM)"):
    start = b * BATCH_SIZE_CLASSIFY
    end = start + BATCH_SIZE_CLASSIFY
    batch = texts_for_llm[start:end]
    results = classify_theme_batch(batch)
    for r in results:
        local_idx = r["index"] - 1  # position WITHIN this batch only (0..len(batch)-1)
        if 0 <= local_idx < len(batch):
            global_idx = need_llm_idx[start + local_idx]
            themes_raw = r.get("themes", ["others"])
            themes_valid = [t for t in themes_raw if t in THEMES] or ["others"]
            primary = r.get("primary_theme", themes_valid[0])
            themes_run[global_idx] = primary if primary in THEMES else themes_valid[0]
            all_themes_run[global_idx] = themes_valid
            confidences_run[global_idx] = r.get("confidence", 0.5)
    time.sleep(SLEEP_BETWEEN_CALLS)

# Safety net in case anything is still None (shouldn't happen)
themes_run = [th if th is not None else "others" for th in themes_run]
all_themes_run = [at if at is not None else ["others"] for at in all_themes_run]

df_run["theme"] = themes_run
df_run["all_themes"] = all_themes_run
df_run["theme_confidence"] = confidences_run
df_run["theme"] = df_run["theme"].fillna("others")
df_run["theme_confidence"] = df_run["theme_confidence"].fillna(0.0)

print("\nTheme distribution (primary):")
display(df_run["theme"].value_counts())

print("\nExamples with multiple themes (from LLM results):")
display(df_run[df_run["all_themes"].apply(len) > 1][["text_clean", "all_themes", "theme"]].head(10))

display(df_run[["text_clean", "theme", "all_themes", "theme_confidence"]].head(10))

df_clean = df_run.copy()
print(f"\n[INFO] df_clean now contains {len(df_clean)} rows")

# -----------------------------------------------------------------------
# SAVE CHECKPOINT
# -----------------------------------------------------------------------
# IMPORTANT: reuses CONFIG["checkpoint_path"] from config.py — the SAME path
# Step 3 checks via RESUME_FROM_THEME. Do not introduce a separate hardcoded
# path here, or the resume logic in your next run will silently break
# (it'll re-download and re-classify everything because it's checking a
# different file than the one you just saved to).
from google.colab import drive
drive.mount('/content/drive')

df_clean.to_pickle(CONFIG["checkpoint_path"])
print(f"[OK] df_clean saved to {CONFIG['checkpoint_path']}, {len(df_clean)} rows")

# STEP 6 — NAMED ENTITY RECOGNITION (locations, institutions, persons)

Runs exclusively on **non-media** rows with **relevant** (non-"others") themes to minimize API costs. Remove the filter below if full dataset NER is required.

⚠️ ACTION REQUIRED: The `FEWSHOT_NER_EXAMPLES` are placeholders. You MUST replace them with real-world examples and entities from your specific topic. Irrelevant examples will degrade extraction accuracy.

Ensure your examples cover high-variations: formal texts, casual/slang social media posts, multi-entity instances, empty entity cases, and negative controls.

Entity categories (location/institution/person) are standard for Indonesian projects, but can be customized in `build_ner_prompt()` if needed (e.g., adding products or brands).

In [ ]:
FEWSHOT_NER_EXAMPLES = """
Extraction examples (learn from this pattern):

Text: "[REPLACE: formal/news-style text mentioning one person, one location, and one institution/event together]"
Answer: {"lokasi": ["[REPLACE: location name]"], "instansi": ["[REPLACE: institution/event name]"], "tokoh": ["[REPLACE: person name]"]}

Text: "[REPLACE: text mentioning two people and one location, with no institution]"
Answer: {"lokasi": ["[REPLACE: location name]"], "instansi": [], "tokoh": ["[REPLACE: person 1]", "[REPLACE: person 2]"]}

Text: "[REPLACE: text with no location/institution/person mentioned at all]"
Answer: {"lokasi": [], "instansi": [], "tokoh": []}
"""
# TIP: keep adding real examples here as you check wrong classification
# results — just like the note in FEWSHOT_THEME_EXAMPLES in Step 4,
# this is a strong candidate to be moved into a growing few-shot bank
# (not a fixed static block) in the future.


def build_ner_prompt(batch_texts):
    numbered = "\n".join(f"{i+1}. {t}" for i, t in enumerate(batch_texts))
    return f"""Extract important entities from each of the following Indonesian texts.
Entity categories:
- lokasi: country/city/place name (example: [REPLACE: your topic's specific location name])
- instansi: institution/organization/official event name (example: [REPLACE: your topic's specific institution/event name])
- tokoh: name of the person mentioned (official, public figure, etc.)

{FEWSHOT_NER_EXAMPLES}

Now extract entities from the following texts. If there are no entities in a specific category,
use an empty array (see the 3rd example above) — DO NOT fabricate entities.

Text:
{numbered}

Answer ONLY with a JSON array, without explanations, without markdown code blocks:
[
  {{"index": 1, "lokasi": ["..."], "instansi": ["..."], "tokoh": ["..."]}},
  {{"index": 2, "lokasi": [], "instansi": [], "tokoh": []}}
]
The number of items MUST exactly match the number of texts ({len(batch_texts)} items)."""


def extract_ner_batch(batch_texts, retry=6):
    prompt = build_ner_prompt(batch_texts)
    for attempt in range(retry):
        try:
            resp = gemini_client.models.generate_content(
                model=GEMINI_MODEL,
                contents=prompt,
                config=types.GenerateContentConfig(
                    response_mime_type="application/json",
                    max_output_tokens=2000,
                ),
            )
            parsed = json.loads(resp.text.strip())
            if len(parsed) != len(batch_texts):
                raise ValueError(f"Number of results ({len(parsed)}) != number of inputs ({len(batch_texts)})")
            return parsed, True
        except Exception as e:
            is_overload = "503" in str(e) or "UNAVAILABLE" in str(e)
            wait = 20 * (attempt + 1) if is_overload else 5 * (attempt + 1)
            print(f"[WARN] NER batch failed (attempt {attempt+1}/{retry}): {e} -> waiting {wait}s")
            time.sleep(wait)
    print(f"[ERROR] This NER batch failed completely after {retry} attempts")
    return [{"index": i + 1, "lokasi": [], "instansi": [], "tokoh": []} for i in range(len(batch_texts))], False


target_idx = df_clean[df_clean["theme"] != "others"].index.tolist()
print(f"Running NER for {len(target_idx)} rows (relevant themes) ...")

texts_all = df_clean["text_clean"].fillna("").tolist()
lokasi_col = [[] for _ in range(len(df_clean))]
instansi_col = [[] for _ in range(len(df_clean))]
tokoh_col = [[] for _ in range(len(df_clean))]
ner_failed_col = [False for _ in range(len(df_clean))]

n_batches = (len(target_idx) + BATCH_SIZE_NER - 1) // BATCH_SIZE_NER
for b in tqdm(range(n_batches), desc="NER entity extraction"):
    idx_batch = target_idx[b * BATCH_SIZE_NER:(b + 1) * BATCH_SIZE_NER]
    text_batch = [texts_all[i] for i in idx_batch]
    results, success = extract_ner_batch(text_batch)
    for r, df_idx in zip(results, idx_batch):
        lokasi_col[df_idx] = r.get("lokasi", [])
        instansi_col[df_idx] = r.get("instansi", [])
        tokoh_col[df_idx] = r.get("tokoh", [])
        ner_failed_col[df_idx] = not success
    time.sleep(SLEEP_BETWEEN_CALLS)

df_clean["entities_lokasi"] = lokasi_col
df_clean["entities_instansi"] = instansi_col
df_clean["entities_tokoh"] = tokoh_col
df_clean["ner_failed"] = ner_failed_col

n_failed = df_clean.loc[target_idx, "ner_failed"].sum()
if n_failed > 0:
    print(f"[WARN] {n_failed} rows failed NER processing after all attempts, their entities being empty does not mean they do not exist")

print("\nNER result examples:")
display(df_clean.loc[target_idx, ["text_clean", "entities_lokasi", "entities_instansi", "entities_tokoh", "ner_failed"]].head(10))

# SETUP 7 — RANKING TOP AUTHORS PER THEME

Calculates author scores for **non-media** accounts based on a weighted combination of post count, engagement, and follower count (weights defined in `CONFIG`, Setup 3).

In [ ]:
df_topic = df_clean[df_clean["theme"] != "others"].copy()
print(f"[INFO] {len(df_topic)} rows remaining after relevant theme filter (media was dropped since Setup 3b)")

for col in [COL_FAVOURITED, COL_RETWEETED]:
    if col in df_topic.columns:
        df_topic[col] = pd.to_numeric(df_topic[col], errors="coerce").fillna(0)
    else:
        df_topic[col] = 0

if COL_FOLLOWERS in df_topic.columns:
    df_topic[COL_FOLLOWERS] = pd.to_numeric(df_topic[COL_FOLLOWERS], errors="coerce").fillna(0)
else:
    df_topic[COL_FOLLOWERS] = 0

df_topic["engagement"] = df_topic[COL_FAVOURITED] + df_topic[COL_RETWEETED]

top_authors_per_theme = {}

for theme in THEMES:
    if theme == "others":
        continue
    sub = df_topic[df_topic["theme"] == theme]
    if sub.empty:
        print(f"[INFO] No data for theme '{theme}'")
        continue

    agg = (
        sub.groupby(COL_AUTHOR_ID)
        .agg(
            post_count=(COL_AUTHOR_ID, "count"),
            total_engagement=("engagement", "sum"),
            followers=(COL_FOLLOWERS, "max"),
        )
        .reset_index()
        .rename(columns={COL_AUTHOR_ID: "author_id"})
    )

    max_post = agg["post_count"].max() or 1
    max_eng = agg["total_engagement"].max() or 1
    max_followers = agg["followers"].max() or 1
    agg["score"] = (
        WEIGHT_POST_COUNT * (agg["post_count"] / max_post)
        + WEIGHT_ENGAGEMENT * (agg["total_engagement"] / max_eng)
        + WEIGHT_FOLLOWERS * (agg["followers"] / max_followers)
    )

    agg = agg.sort_values("score", ascending=False).head(TOP_N_AUTHORS_PER_THEME)
    agg.insert(0, "theme", theme)
    agg["rank"] = range(1, len(agg) + 1)
    top_authors_per_theme[theme] = agg

    print(f"\n=== Top Author: {theme} ===")
    display(agg[["rank", "author_id", "post_count", "total_engagement", "followers", "score"]])

df_top_authors = (
    pd.concat(top_authors_per_theme.values(), ignore_index=True)
    if top_authors_per_theme else pd.DataFrame(columns=["theme", "author_id", "post_count", "total_engagement", "followers", "score", "rank"])
)

# SETUP 8 — SUMMARY & SENTIMENT PER TOP AUTHOR (LLM)

Sends all posts by each top author for a given theme to the LLM for summarization and overall sentiment determination. Per-post monitoring sentiment (`Sentiment`) is provided strictly as supplementary context, not as ground truth.

⚠️ TOPIC-SPECIFIC: `SENTIMENT_OPTIONS` ("positive"/"negative"/"controversial") is a standard 3-class scheme. Customize these categories and their definitions in `build_summary_prompt()` if your project requires an alternative schema.

### STEP 8a — SUMMARIZATION FUNCTIONS

In [ ]:
SENTIMENT_OPTIONS = ["positive", "negative", "controversial"]

def build_summary_prompt(author_label, theme, texts, tool_sentiment_note=""):
    joined = "\n".join(f"- {t}" for t in texts)
    context_note = ""
    if tool_sentiment_note:
        context_note = (
            f"\nAs an additional reference (not an absolute baseline), the social media monitoring tool "
            f"previously labeled the per-post sentiment for this account with the distribution: "
            f"{tool_sentiment_note}. Use this only as a consideration; "
            f"the final decision remains based on the text content you read yourself.\n"
        )
    return f"""Here is a collection of tweets from the account "{author_label}" about the topic "{theme}":

{joined}
{context_note}
Your tasks:
1. Summarize in 2-3 sentences the main view/narrative conveyed by this account regarding the topic.
2. Determine the OVERALL sentiment of this account towards the topic, choose ONE from: {", ".join(SENTIMENT_OPTIONS)}.
   - "positive" is used if the account supports/praises this topic.
   - "negative" is used if the account criticizes/opposes this topic.
   - "controversial" is used if the account's opinion triggers debate/pros-cons, conveys controversial claims, or mixes praise and criticism at the same time.
3. Provide a brief reason (1 sentence) for that sentiment.

Answer ONLY with JSON, without additional explanations, without markdown code blocks:
{{"summary": "...", "sentiment": "...", "reason": "..."}}"""


def summarize_author(author_label, theme, texts, tool_sentiment_note="", retry=3):
    prompt = build_summary_prompt(author_label, theme, texts, tool_sentiment_note)
    for attempt in range(retry):
        try:
            resp = gemini_client.models.generate_content(
                model=GEMINI_MODEL,
                contents=prompt,
                config=types.GenerateContentConfig(
                    response_mime_type="application/json",
                    max_output_tokens=2000,
                ),
            )
            parsed = json.loads(resp.text.strip())
            if parsed.get("sentiment") not in SENTIMENT_OPTIONS:
                parsed["sentiment"] = "controversial"
            return parsed
        except Exception as e:
            wait = 5 * (attempt + 1)
            print(f"[WARN] failed to summarize {author_label} (attempt {attempt+1}/{retry}): {e} -> waiting {wait}s")
            time.sleep(wait)
    return {"summary": "(failed to summarize automatically)", "sentiment": "controversial", "reason": "API error"}

print("Summarization functions ready.")

### SETUP 8b — RUN SUMMARIZATION FOR ALL TOP AUTHORS AND EZPORT TO EXCEL

In [ ]:
pd.set_option("display.max_colwidth", None)   # do not truncate long text column content
pd.set_option("display.max_rows", None)       # display all rows (optional, be careful if there's a lot of data)

results = []

for _, row in df_top_authors.iterrows():
    author_id = row["author_id"]
    theme = row["theme"]

    author_posts_df = df_topic[(df_topic[COL_AUTHOR_ID] == author_id) & (df_topic["theme"] == theme)]
    posts = author_posts_df["text_clean"].dropna().tolist()

    tool_sentiment_note = ""
    if COL_SENTIMENT in author_posts_df.columns:
        counts = author_posts_df[COL_SENTIMENT].dropna().value_counts()
        if not counts.empty:
            total = counts.sum()
            tool_sentiment_note = ", ".join(f"{label} {round(100 * n / total)}%" for label, n in counts.items())

    posts = posts[:MAX_POSTS_PER_AUTHOR_SUMMARY]
    if not posts:
        continue

    print(f"Summarizing @{author_id} | theme={theme} | {len(posts)} posts ...")
    result = summarize_author(author_id, theme, posts, tool_sentiment_note)

    results.append({
        "theme": theme,
        "rank": int(row["rank"]),
        "author_id": author_id,
        "score": round(row["score"], 2),
        "summary": result.get("summary", ""),
        "sentiment": result.get("sentiment", "controversial"),
        "reason": result.get("reason", ""),
    })

df_summary = pd.DataFrame(results)

print("\n=== Final result: summary & sentiment per top author ===")
display(df_summary)

# --- Export to Excel ---
file_name = "top_author_analysis_results.xlsx"
df_summary.to_excel(file_name, index=False)
print(f"\n[OK] Data successfully exported to {file_name}")

from google.colab import files
files.download(file_name)